#### Load dependencies and dataset

In [15]:
import pandas as pd
import numpy as np

from time import time

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.base import clone

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

In [3]:
df = pd.read_csv("../data/processed/cleaned_healthcare.csv")

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (54966, 15)


,name,age,gender,blood_type,medical_condition,date_of_admission,doctor,hospital,insurance_provider,billing_amount,room_number,admission_type,discharge_date,medication,test_results
0,Bobby Jackson,30,Male,B-,Cancer,2024-01-31,Matthew Smith,Sons and Miller,Blue Cross,18856.281306,328,Urgent,2024-02-02,Paracetamol,Normal
1,Leslie Terry,62,Male,A+,Obesity,2019-08-20,Samantha Davies,Kim Inc,Medicare,33643.327287,265,Emergency,2019-08-26,Ibuprofen,Inconclusive
2,Danny Smith,76,Female,A-,Obesity,2022-09-22,Tiffany Mitchell,Cook PLC,Aetna,27955.096079,205,Emergency,2022-10-07,Aspirin,Normal
3,Andrew Watts,28,Female,O+,Diabetes,2020-11-18,Kevin Wells,"Hernandez Rogers and Vang,",Medicare,37909.782410,450,Elective,2020-12-18,Ibuprofen,Abnormal
4,Adrienne Bell,43,Female,AB+,Cancer,2022-09-19,Kathleen Hanna,White-White,Aetna,14238.317814,458,Urgent,2022-10-09,Penicillin,Abnormal


#### STEP 2: Define X and y

In [4]:
feature_cols = [
    "age",
    "gender",
    "blood_type",
    "medical_condition",
    "insurance_provider",
    "admission_type",
    "medication",
]

target_col = "test_results"

X = df[feature_cols].copy()
y = df[target_col].copy()

print("Features shape:", X.shape)
print("Target shape:", y.shape)

Features shape: (54966, 7)
Target shape: (54966,)


#### STEP 3: Split into Train, Test(validation) sets

In [6]:
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)


#### STEP 4: Build Preprocessing pipeline

In [7]:
numeric_features = ["age"]

categorical_features = [
    "gender",
    "blood_type",
    "medical_condition",
    "insurance_provider",
    "admission_type",
    "medication",
]

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

#### STEP 5: Train Dummy classifier

In [8]:
dummy_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", DummyClassifier(strategy="most_frequent")),
    ]
)

dummy_model.fit(X_train, y_train)

y_pred_dummy = dummy_model.predict(X_val)

print("Dummy Accuracy:", accuracy_score(y_val, y_pred_dummy))
print("Dummy Macro F1:", f1_score(y_val, y_pred_dummy, average="macro"))

Dummy Accuracy: 0.3354557031107877
Dummy Macro F1: 0.16746129046905509


In [10]:
print(classification_report(y_val, y_pred_dummy, zero_division=0))

              precision    recall  f1-score   support

    Abnormal       0.34      1.00      0.50      3688
Inconclusive       0.00      0.00      0.00      3640
      Normal       0.00      0.00      0.00      3666

    accuracy                           0.34     10994
   macro avg       0.11      0.33      0.17     10994
weighted avg       0.11      0.34      0.17     10994



#### STEP 6: Model Training

In [23]:

# Define the models
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        random_state=42,
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1,
    ),
    "XGBoost": XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        objective="multi:softprob",
        eval_metric="mlogloss",
        random_state=42,
        n_jobs=-1,
    ),
}

# Initialize result containers
results = []
trained_models = {}
confusion_matrices = {}
class_reports = {}

# Encode target labels for XGBoost
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_val_encoded = label_encoder.transform(y_val)

# Use one consistent class order for all models
class_labels = label_encoder.classes_

# Train, predict, and evaluate each model
for name, model in models.items():
    print(f"Training {name}...")

    pipeline = Pipeline(
        steps=[
            ("preprocessor", clone(preprocessor)),
            ("classifier", model),
        ]
    )

    start_time = time()

    if name == "XGBoost":
        pipeline.fit(X_train, y_train_encoded)
        y_pred_encoded = pipeline.predict(X_val).astype(int)
        y_pred = label_encoder.inverse_transform(y_pred_encoded)
    else:
        pipeline.fit(X_train, y_train)
        y_pred = pipeline.predict(X_val)

    training_seconds = time() - start_time

    # Calculate and store evaluation results for EVERY model
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_val, y_pred),
        "Precision (Macro)": precision_score(
            y_val, y_pred, average="macro", zero_division=0
        ),
        "Recall (Macro)": recall_score(
            y_val, y_pred, average="macro", zero_division=0
        ),
        "F1 (Macro)": f1_score(
            y_val, y_pred, average="macro", zero_division=0
        ),
        "Training + Prediction (seconds)": round(training_seconds, 2),
    })

    # Store the fitted pipeline and detailed evaluation results
    trained_models[name] = pipeline

    confusion_matrices[name] = confusion_matrix(
        y_val,
        y_pred,
        labels=class_labels,
    )

    class_reports[name] = classification_report(
        y_val,
        y_pred,
        labels=class_labels,
        target_names=class_labels,
        zero_division=0,
    )

    print(f"Completed {name}. Results collected: {len(results)}")

# Compare all models
results_df = (
    pd.DataFrame(results)
    .sort_values("F1 (Macro)", ascending=False)
    .reset_index(drop=True)
)

display(results_df.round(4))

# Display detailed evaluation results
for name in results_df["Model"]:
    print(f"\n{'=' * 60}")
    print(name)
    print("Class order:", list(class_labels))
    print("Confusion Matrix:")
    print(confusion_matrices[name])
    print("\nClassification Report:")
    print(class_reports[name])


Training Logistic Regression...
Completed Logistic Regression. Results collected: 1
Training Random Forest...
Completed Random Forest. Results collected: 2
Training XGBoost...
Completed XGBoost. Results collected: 3


,Model,Accuracy,Precision (Macro),Recall (Macro),F1 (Macro),Training + Prediction (seconds)
0,Random Forest,0.3850,0.3851,0.3850,0.3850,124.83
1,XGBoost,0.3403,0.3403,0.3402,0.3402,5.57
2,Logistic Regression,0.3344,0.3351,0.3340,0.3306,0.88



Random Forest
Class order: ['Abnormal', 'Inconclusive', 'Normal']
Confusion Matrix:
[[1450 1096 1142]
 [1130 1373 1137]
 [1186 1070 1410]]

Classification Report:
              precision    recall  f1-score   support

    Abnormal       0.39      0.39      0.39      3688
Inconclusive       0.39      0.38      0.38      3640
      Normal       0.38      0.38      0.38      3666

    accuracy                           0.39     10994
   macro avg       0.39      0.38      0.38     10994
weighted avg       0.39      0.39      0.39     10994


XGBoost
Class order: ['Abnormal', 'Inconclusive', 'Normal']
Confusion Matrix:
[[1317 1170 1201]
 [1255 1219 1166]
 [1304 1157 1205]]

Classification Report:
              precision    recall  f1-score   support

    Abnormal       0.34      0.36      0.35      3688
Inconclusive       0.34      0.33      0.34      3640
      Normal       0.34      0.33      0.33      3666

    accuracy                           0.34     10994
   macro avg       0.34  

## Model Evaluation and Selection

We evaluated Logistic Regression, Random Forest, and XGBoost against a Dummy Classifier baseline using a stratified validation set.

Random Forest achieved the best overall performance, with **38.50% accuracy and a macro F1-score of 0.3850**. It outperformed XGBoost (macro F1: 0.3402) and Logistic Regression (macro F1: 0.3306).

The confusion matrix and classification report show that Random Forest performs similarly across the three classes, although its predictions remain limited. Its macro F1-score indicates that classification performance is still modest.

We selected Random Forest as the best-performing candidate among the models tested. Extensive hyperparameter tuning was not pursued because the immediate project objective is to demonstrate an end-to-end machine-learning workflow, including training, evaluation, model persistence, API integration, and deployment.

**Limitation:** The dataset contains no explicit laboratory measurements or other direct test-related indicators. Consequently, this model should be treated as a demonstration baseline rather than a clinically validated system for predicting medical test results.

#### STEP 7: Save models

In [27]:
best_model = trained_models["Random Forest"]

In [29]:
from pathlib import Path
import joblib

models_dir = Path("../models")
models_dir.mkdir(parents=True, exist_ok=True)

# Save the fitted pipeline
model_path = models_dir / "model.joblib"
joblib.dump(best_model, model_path)

print(f"Model saved to: {model_path.resolve()}")

Model saved to: C:\Users\Prof\Desktop\Projects\healthcare-analytics-system\models\model.joblib
